# Quran Muaalem dataset exploration

**Scope:** metadata and a small representative sample only. This notebook does not preprocess or train. It uses the Hugging Face datasets-server JSON API and fetches only selected short WAV assets, avoiding the approximately 97 GB dataset payload.

Run from the repository root with Python 3.10+ and `requests`, `pandas`, `numpy`, `soundfile`, `IPython`, and Jupyter installed. For example: `python -m pip install requests pandas numpy soundfile ipython jupyter nbclient`. Then run `jupyter nbconvert --to notebook --execute notebooks/01_dataset_exploration.ipynb --inplace`. The remote API is live, so counts or samples may change in later runs.

## 1. Dataset information

The dataset card describes annotated Quran recitation audio and links it to the Quran Muaalem project. The API calls below record the current server metadata. The Hugging Face viewer page reports a row post-processing error for at least one set; this notebook records endpoint failures instead of hiding them.

In [1]:
import requests, pandas as pd, numpy as np, soundfile as sf, io, json, math
from IPython.display import Audio, display

DATASET = "obadx/muaalem-annotated-v3"
API = "https://datasets-server.huggingface.co"

def api_json(endpoint, **params):
    response = requests.get(f"{API}/{endpoint}", params={"dataset": DATASET, **params}, timeout=90)
    response.raise_for_status()
    return response.json()

size_payload = api_json("size")["size"]
splits_payload = api_json("splits")
print("Dataset:", DATASET)
print("Rows reported:", size_payload["dataset"]["num_rows"])
print("Original/parquet file bytes:", size_payload["dataset"]["num_bytes_original_files"])
print("Split entries:", len(splits_payload["splits"]))

Dataset: obadx/muaalem-annotated-v3
Rows reported: 286586
Original/parquet file bytes: 97341255406
Split entries: 29


## 2. Available subsets and splits

`size` reports per-configuration row counts and columns; `splits` reports the available split names. A split count is the server-reported count, not a count obtained by downloading examples.

In [2]:
configs = pd.DataFrame(size_payload["configs"])[["config", "num_rows", "num_columns", "num_bytes_parquet_files"]]
splits = pd.DataFrame(splits_payload["splits"])[["config", "split"]]
subset_summary = configs.merge(splits.groupby("config")["split"].agg(lambda x: ", ".join(x)).rename("splits"), on="config", how="left")
display(subset_summary.sort_values("config").reset_index(drop=True))
print("Configurations:", len(configs), "| Total rows:", int(configs.num_rows.sum()))

,config,num_rows,num_columns,num_bytes_parquet_files,splits
0,moshaf_0.0,9133,24,3230445287,train
1,moshaf_0.1,10764,24,4591995504,train
2,moshaf_0.2,9971,24,5634459149,train
3,moshaf_0.3,12604,24,4243397460,train
4,moshaf_1.0,10939,24,3265981122,train
5,moshaf_11.0,10363,24,2770216689,train
6,moshaf_12.0,9880,24,3840493442,train
7,moshaf_13.0,9377,24,3907411764,train
8,moshaf_19.0,11278,24,3473021134,train
9,moshaf_2.0,9942,24,5785050169,train


Configurations: 29 | Total rows: 286586


## 3. Dataset schema

The API feature declarations are the source for the types shown here. For each recitation configuration where first-row processing succeeds, compare the feature declarations. A failure is reported explicitly. This matters because the server cannot currently post-process first rows for every configuration.

In [3]:
schema_results = {}
schema_errors = {}
for config in configs.config:
    try:
        payload = api_json("first-rows", config=config, split="train")
        schema_results[config] = payload["features"]
    except Exception as exc:
        schema_errors[config] = str(exc)

schema_fingerprints = {}
for config, features in schema_results.items():
    canonical = json.dumps(features, ensure_ascii=False, sort_keys=True)
    schema_fingerprints.setdefault(canonical, []).append(config)
print("Distinct verified schemas:", len(schema_fingerprints))
for i, (schema, members) in enumerate(schema_fingerprints.items(), start=1):
    print(f"Schema {i}: {len(members)} configs; {members}")
print("First-row schema errors:", schema_errors)

# Display the complete verified recitation schema (if accessible) and both metadata schemas.
for wanted in ["moshaf_0.1", "moshaf_metadata", "reciters_metadata"]:
    if wanted in schema_results:
        print("\n", wanted)
        display(pd.DataFrame([{"column": f["name"], "type": json.dumps(f["type"], ensure_ascii=False)} for f in schema_results[wanted]]))

Distinct verified schemas: 3
Schema 1: 21 configs; ['moshaf_0.1', 'moshaf_0.2', 'moshaf_0.3', 'moshaf_1.0', 'moshaf_12.0', 'moshaf_13.0', 'moshaf_2.0', 'moshaf_2.1', 'moshaf_22.0', 'moshaf_24.0', 'moshaf_25.0', 'moshaf_26.1', 'moshaf_27.0', 'moshaf_28.0', 'moshaf_29.0', 'moshaf_3.0', 'moshaf_4.0', 'moshaf_6.0', 'moshaf_7.0', 'moshaf_8.0', 'moshaf_9.0']
Schema 2: 1 configs; ['moshaf_metadata']
Schema 3: 1 configs; ['reciters_metadata']
First-row schema errors: {'moshaf_0.0': '500 Server Error: Internal Server Error for url: https://datasets-server.huggingface.co/first-rows?dataset=obadx%2Fmuaalem-annotated-v3&config=moshaf_0.0&split=train', 'moshaf_11.0': '500 Server Error: Internal Server Error for url: https://datasets-server.huggingface.co/first-rows?dataset=obadx%2Fmuaalem-annotated-v3&config=moshaf_11.0&split=train', 'moshaf_19.0': '500 Server Error: Internal Server Error for url: https://datasets-server.huggingface.co/first-rows?dataset=obadx%2Fmuaalem-annotated-v3&config=moshaf_1

,column,type
0,audio,"{""sampling_rate"": 16000, ""_type"": ""Audio""}"
1,segment_index,"{""dtype"": ""string"", ""_type"": ""Value""}"
2,tarteel_transcript,"{""feature"": {""dtype"": ""string"", ""_type"": ""Valu..."
3,moshaf_id,"{""dtype"": ""string"", ""_type"": ""Value""}"
4,moshaf_name,"{""dtype"": ""string"", ""_type"": ""Value""}"
5,reciter_id,"{""dtype"": ""int32"", ""_type"": ""Value""}"
6,reciter_arabic_name,"{""dtype"": ""string"", ""_type"": ""Value""}"
7,reciter_english_name,"{""dtype"": ""string"", ""_type"": ""Value""}"
8,sura_or_aya_index,"{""dtype"": ""string"", ""_type"": ""Value""}"
9,index_type,"{""dtype"": ""string"", ""_type"": ""Value""}"



 moshaf_metadata


,column,type
0,comments,"{""dtype"": ""string"", ""_type"": ""Value""}"
1,total_size_mb,"{""dtype"": ""float32"", ""_type"": ""Value""}"
2,path,"{""dtype"": ""string"", ""_type"": ""Value""}"
3,madd_monfasel_len,"{""dtype"": ""int32"", ""_type"": ""Value""}"
4,irkab_maana,"{""dtype"": ""string"", ""_type"": ""Value""}"
5,yaa_ataan,"{""dtype"": ""string"", ""_type"": ""Value""}"
6,madd_yaa_alayn_alharfy,"{""dtype"": ""int32"", ""_type"": ""Value""}"
7,reciter_id,"{""dtype"": ""int32"", ""_type"": ""Value""}"
8,name,"{""dtype"": ""string"", ""_type"": ""Value""}"
9,rewaya,"{""dtype"": ""string"", ""_type"": ""Value""}"



 reciters_metadata


,column,type
0,arabic_name,"{""dtype"": ""string"", ""_type"": ""Value""}"
1,moshaf_set_ids,"{""feature"": {""dtype"": ""string"", ""_type"": ""Valu..."
2,english_name,"{""dtype"": ""string"", ""_type"": ""Value""}"
3,country_code,"{""dtype"": ""string"", ""_type"": ""Value""}"
4,id,"{""dtype"": ""int32"", ""_type"": ""Value""}"


## 4. Sample inspection

The `rows` endpoint is requested with a small `length`. Audio is represented by an asset URL and is not printed as an array or encoded payload. `moshaf_0.0` is tried as a requested representative as it is one of the HF viewer configurations; the error, if still present, is kept as a finding. The succeeding samples use different mushaf sets/reciters.

In [4]:
def get_rows(config, offset=0, length=5):
    try:
        return api_json("rows", config=config, split="train", offset=offset, length=length)["rows"]
    except Exception as rows_exc:
        # The rows endpoint has intermittently failed for some configurations; use the
        # first-rows endpoint as a small fallback and preserve any failure if both fail.
        try:
            return api_json("first-rows", config=config, split="train")["rows"][:length]
        except Exception as first_exc:
            raise RuntimeError(f"rows failed: {rows_exc}; first-rows failed: {first_exc}") from first_exc

sample_configs = ["moshaf_0.0", "moshaf_0.1", "moshaf_1.0", "moshaf_26.1"]
representative_rows = {}
sample_errors = {}
for config in sample_configs:
    try:
        representative_rows[config] = get_rows(config, length=5)
    except Exception as exc:
        sample_errors[config] = str(exc)
print("Sample errors:", sample_errors)

def brief_row(config, item):
    r = item["row"]
    audio = r.get("audio")
    audio_ref = audio[0] if isinstance(audio, list) and audio else audio
    return {"config": config, "row_idx": item["row_idx"], "segment_index": r.get("segment_index"),
            "moshaf_id": r.get("moshaf_id"), "reciter": r.get("reciter_english_name"),
            "index_type": r.get("index_type"), "sura_or_aya_index": r.get("sura_or_aya_index"),
            "duration_seconds": r.get("duration_seconds"), "sample_rate": r.get("sample_rate"),
            "audio_type": audio_ref.get("type") if isinstance(audio_ref, dict) else None,
            "imlaey": r.get("imlaey"), "uthmani": r.get("uthmani"),
            "has_quran": r.get("has_quran"), "start_span": r.get("start_span"), "end_span": r.get("end_span"),
            "match_ratio": r.get("match_ratio"), "phonemes": r.get("phonemes"),
            "sifat_count": len(r.get("sifat") or [])}

sample_frame = pd.DataFrame([brief_row(cfg, row) for cfg, rows in representative_rows.items() for row in rows])
display(sample_frame.head(12))

Sample errors: {}


,config,row_idx,segment_index,moshaf_id,reciter,index_type,sura_or_aya_index,duration_seconds,sample_rate,audio_type,imlaey,uthmani,has_quran,start_span,end_span,match_ratio,phonemes,sifat_count
0,moshaf_0.0,0,001.0000,0.0,Mahmoud Khalil Al-Husari,sura,001,4.150,16000,audio/wav,أَعُوذُ بِاللَّهِ مِنَ الشَّيْطَانِ الرَّجِيمِ,أَعُوذُ بِٱللَّهِ مِنَ ٱلشَّيْطَانِ ٱلرَّجِيمِ,False,None,None,1.000000,ءَعُۥۥذُ بِللَااهِ مِنَ ششَيطَاانِ ررَجِۦۦم,19
1,moshaf_0.0,1,001.0001,0.0,Mahmoud Khalil Al-Husari,sura,001,3.620,16000,audio/wav,بِسْمِ اللَّهِ الرَّحْمَٰنِ الرَّحِيمِ,بِسْمِ ٱللَّهِ ٱلرَّحْمَـٰنِ ٱلرَّحِيمِ,True,"{'sura_idx': 1, 'aya_idx': 1, 'imlaey': 0, 'ut...","{'sura_idx': 1, 'aya_idx': 1, 'imlaey': 4, 'ut...",1.000000,بِسمِ للَااهِ ررَحمَاانِ ررَحِۦۦم,15
2,moshaf_0.0,2,001.0002,0.0,Mahmoud Khalil Al-Husari,sura,001,3.980,16000,audio/wav,الْحَمْدُ لِلَّهِ رَبِّ الْعَالَمِينَ,ٱلْحَمْدُ لِلَّهِ رَبِّ ٱلْعَـٰلَمِينَ,True,"{'sura_idx': 1, 'aya_idx': 2, 'imlaey': 0, 'ut...","{'sura_idx': 1, 'aya_idx': 2, 'imlaey': 4, 'ut...",1.000000,ءَلحَمدُ لِللَااهِ رَببِ لعَاالَمِۦۦن,18
3,moshaf_0.0,3,001.0003,0.0,Mahmoud Khalil Al-Husari,sura,001,3.020,16000,audio/wav,الرَّحْمَٰنِ الرَّحِيمِ,ٱلرَّحْمَـٰنِ ٱلرَّحِيمِ,True,"{'sura_idx': 1, 'aya_idx': 3, 'imlaey': 0, 'ut...","{'sura_idx': 1, 'aya_idx': 3, 'imlaey': 2, 'ut...",1.000000,ءَررَحمَاانِ ررَحِۦۦم,10
4,moshaf_0.0,4,001.0004,0.0,Mahmoud Khalil Al-Husari,sura,001,3.140,16000,audio/wav,مَالِكِ يَوْمِ الدِّينِ,مَـٰلِكِ يَوْمِ ٱلدِّينِ,True,"{'sura_idx': 1, 'aya_idx': 4, 'imlaey': 0, 'ut...","{'sura_idx': 1, 'aya_idx': 4, 'imlaey': 3, 'ut...",1.000000,مَاالِكِ يَومِ ددِۦۦن,10
5,moshaf_0.1,0,001.0000,0.1,Mahmoud Khalil Al-Husari,sura,001,6.060,16000,audio/wav,أَعُوذُ بِاللَّهِ مِنَ الشَّيْطَانِ الرَّجِيمِ,أَعُوذُ بِٱللَّهِ مِنَ ٱلشَّيْطَانِ ٱلرَّجِيمِ,False,None,None,0.956522,ءَعُۥۥذُ بِللَااهِ مِنَ ششَيطَاانِ ررَجِۦۦۦۦم,19
6,moshaf_0.1,1,001.0001,0.1,Mahmoud Khalil Al-Husari,sura,001,6.020,16000,audio/wav,بِسْمِ اللَّهِ الرَّحْمَٰنِ الرَّحِيمِ,بِسْمِ ٱللَّهِ ٱلرَّحْمَـٰنِ ٱلرَّحِيمِ,True,"{'sura_idx': 1, 'aya_idx': 1, 'imlaey': 0, 'ut...","{'sura_idx': 1, 'aya_idx': 1, 'imlaey': 4, 'ut...",1.000000,بِسمِ للَااهِ ررَحمَاانِ ررَحِۦۦۦۦم,15
7,moshaf_0.1,2,001.0002,0.1,Mahmoud Khalil Al-Husari,sura,001,6.280,16000,audio/wav,الْحَمْدُ لِلَّهِ رَبِّ الْعَالَمِينَ,ٱلْحَمْدُ لِلَّهِ رَبِّ ٱلْعَـٰلَمِينَ,True,"{'sura_idx': 1, 'aya_idx': 2, 'imlaey': 0, 'ut...","{'sura_idx': 1, 'aya_idx': 2, 'imlaey': 4, 'ut...",1.000000,ءَلحَمدُ لِللَااهِ رَببِ لعَاالَمِۦۦۦۦن,18
8,moshaf_0.1,3,001.0003,0.1,Mahmoud Khalil Al-Husari,sura,001,5.075,16000,audio/wav,الرَّحْمَٰنِ الرَّحِيمِ,ٱلرَّحْمَـٰنِ ٱلرَّحِيمِ,True,"{'sura_idx': 1, 'aya_idx': 3, 'imlaey': 0, 'ut...","{'sura_idx': 1, 'aya_idx': 3, 'imlaey': 2, 'ut...",1.000000,ءَررَحمَاانِ ررَحِۦۦۦۦم,10
9,moshaf_0.1,4,001.0004,0.1,Mahmoud Khalil Al-Husari,sura,001,5.120,16000,audio/wav,مَالِكِ يَوْمِ الدِّينِ,مَـٰلِكِ يَوْمِ ٱلدِّينِ,True,"{'sura_idx': 1, 'aya_idx': 4, 'imlaey': 0, 'ut...","{'sura_idx': 1, 'aya_idx': 4, 'imlaey': 3, 'ut...",1.000000,مَاالِكِ يَومِ ددِۦۦۦۦن,10


## 5. Audio inspection

The feature declaration says audio sampling rate is 16,000 Hz. The stored WAV bytes from selected records are independently decoded with `soundfile` to verify actual rate, channel count, duration, finite samples, and peak level. Audio widgets allow listening in a notebook UI.

In [5]:
audio_inspection = []
for config in ["moshaf_0.1", "moshaf_1.0"]:
    if config not in representative_rows or not representative_rows[config]:
        continue
    row = representative_rows[config][0]["row"]
    audio_ref = row["audio"][0] if isinstance(row["audio"], list) else row["audio"]
    response = requests.get(audio_ref["src"], timeout=90)
    response.raise_for_status()
    waveform, actual_rate = sf.read(io.BytesIO(response.content), dtype="float32", always_2d=True)
    duration = len(waveform) / actual_rate
    audio_inspection.append({"config": config, "segment_index": row["segment_index"], "http_status": response.status_code,
        "bytes": len(response.content), "declared_audio_type": audio_ref.get("type"), "decoded_rate": actual_rate,
        "channels": waveform.shape[1], "duration_seconds": duration,
        "metadata_duration_seconds": row["duration_seconds"], "finite": bool(np.isfinite(waveform).all()),
        "peak_abs": float(np.max(np.abs(waveform)))})
    print(config, row["segment_index"], "decoded audio")
    display(Audio(waveform.T, rate=actual_rate))
display(pd.DataFrame(audio_inspection))

moshaf_0.1 001.0000 decoded audio


moshaf_1.0 001.0000 decoded audio


,config,segment_index,http_status,bytes,declared_audio_type,decoded_rate,channels,duration_seconds,metadata_duration_seconds,finite,peak_abs
0,moshaf_0.1,001.0000,200,193964,audio/wav,16000,1,6.06,6.06,True,0.591248
1,moshaf_1.0,001.0000,200,156844,audio/wav,16000,1,4.90,4.90,True,0.474884


## 6. Annotation inspection

A recitation row includes Tarteel transcript output, Imlaey text, Uthmani text, phonetic-script text, and a list of per-phoneme `sifat` records. Compare the lists and lengths without treating those fields as interchangeable. The sample below also demonstrates that non-Quran material such as isti`aatha can appear in the same segment table, flagged by `has_quran=False`.

In [ ]:
for config in ["moshaf_0.1", "moshaf_1.0"]:
    if config not in representative_rows or not representative_rows[config]:
        print("No accessible sample for", config)
        continue
    row = representative_rows[config][0]["row"]
    print("\n", config, row["segment_index"], "has_quran=", row["has_quran"])
    print("tarteel_transcript:", row["tarteel_transcript"])
    print("imlaey:", row["imlaey"])
    print("uthmani:", row["uthmani"])
    print("phonemes:", row["phonemes"])
    print("sifat entries:", len(row["sifat"] or []), "first:", (row["sifat"] or [None])[0])

# Show a Quran-containing sample, where span coordinates are populated.
for config in ["moshaf_0.1", "moshaf_1.0"]:
    row = next((x["row"] for x in representative_rows.get(config, []) if x["row"].get("has_quran")), None)
    if row:
        print("\nQuran sample", config, row["segment_index"], row["imlaey"])
        print("span start/end:", row["start_span"], row["end_span"])
        print("phonemes:", row["phonemes"])
        print("sifat count:", len(row["sifat"] or []))


 moshaf_0.1 001.0000 has_quran= False
tarteel_transcript: ['عُوذُ بِاللَّهِ مِنَ الشَّيْطَانِ الرَّجِيمِ']
imlaey: أَعُوذُ بِاللَّهِ مِنَ الشَّيْطَانِ الرَّجِيمِ
uthmani: أَعُوذُ بِٱللَّهِ مِنَ ٱلشَّيْطَانِ ٱلرَّجِيمِ
phonemes: ءَعُۥۥذُ بِللَااهِ مِنَ ششَيطَاانِ ررَجِۦۦۦۦم
sifat entries: 19 first: {'ghonna': 'not_maghnoon', 'hams_or_jahr': 'jahr', 'istitala': 'not_mostateel', 'itbaq': 'monfateh', 'phonemes': 'ءَ', 'qalqla': 'not_moqalqal', 'safeer': 'no_safeer', 'shidda_or_rakhawa': 'shadeed', 'tafashie': 'not_motafashie', 'tafkheem_or_taqeeq': 'moraqaq', 'tikraar': 'not_mokarar'}

 moshaf_1.0 001.0000 has_quran= False
tarteel_transcript: ['رَوْضُ بِاللَّهِ مِنَ الشَّيْطَانِ الرَّجِيمِ']
imlaey: أَعُوذُ بِاللَّهِ مِنَ الشَّيْطَانِ الرَّجِيمِ
uthmani: أَعُوذُ بِٱللَّهِ مِنَ ٱلشَّيْطَانِ ٱلرَّجِيمِ
phonemes: ءَعُۥۥذُ بِللَااهِ مِنَ ششَيطَاانِ ررَجِۦۦم
sifat entries: 19 first: {'ghonna': 'not_maghnoon', 'hams_or_jahr': 'jahr', 'istitala': 'not_mostateel', 'itbaq': 'monfateh', 'phonemes':

## 7. Reciter and recording metadata

Mushaf metadata describes a recording set (reciter, recitation options, source, completeness/annotation flags, and summary fields). Reciter metadata maps reciter IDs to names, country codes, and associated mushaf set IDs. A few rows are displayed only; the full metadata tables remain in the remote dataset.

In [7]:
for config in ["moshaf_metadata", "reciters_metadata"]:
    payload = api_json("first-rows", config=config, split="train")
    rows = [x["row"] for x in payload["rows"]]
    print(config, "feature count", len(payload["features"]))
    display(pd.DataFrame(rows).head(3))

moshaf_metadata feature count 57


,comments,total_size_mb,path,madd_monfasel_len,irkab_maana,yaa_ataan,madd_yaa_alayn_alharfy,reciter_id,name,rewaya,...,raa_alqitr,sakt_maleeyah,alias_start_sec,ghonna_lam_and_raa,madd_mottasel_len,saken_before_hamz,num_recitations,id,total_duraion_minutes,meem_mokhfah
0,,1602.722534,dataset/0.0,2,idgham,wasl,4,0,المصحف المرتل بقصر المنفصل,hafs,...,wasl,waqf,0.0,no_ghonna,4,tahqeek,114,0.0,1741.849609,ikhfaa
1,,2327.563477,dataset/0.1,4,idgham,wasl,6,0,المصحف المرتل بمد المنفصل,hafs,...,wasl,waqf,0.0,no_ghonna,4,tahqeek,114,0.1,2568.969482,ikhfaa
2,,3149.553467,dataset/0.2,2,idgham,wasl,2,0,المصحف المجود بقصر المنفصل,hafs,...,wasl,waqf,0.0,no_ghonna,4,tahqeek,114,0.2,3724.167725,ikhfaa


reciters_metadata feature count 5


,arabic_name,moshaf_set_ids,english_name,country_code,id
0,محمود خليل الحصري,"[0.0, 0.3, 0.2, 0.1]",Mahmoud Khalil Al-Husari,EG,0
1,محمد صديق المنشاوي,[1.0],Mohammed Sidiq Al-Minshawi,EG,1
2,عبد الباسط عبد الصمد,"[2.0, 2.1]",Abdul Basit Abdus Samad,EG,2


## 8. Data quality checks

These checks apply only to the fetched sample rows, not to the whole dataset. They look for null/empty annotations, duplicate IDs inside the sample, duration inconsistencies, and inconsistent metadata sample rates. Audio decode was separately checked above. A wider audit requires iterating all rows and reading audio, which is intentionally outside this exploration.

In [8]:
if not sample_frame.empty:
    print("Rows inspected:", len(sample_frame))
    print("Duplicate (config, segment_index):", int(sample_frame.duplicated(["config", "segment_index"]).sum()))
    for col in ["imlaey", "uthmani", "phonemes"]:
        values = sample_frame[col]
        print(f"{col}: null={int(values.isna().sum())}, empty={int((values.fillna('').astype(str).str.strip() == '').sum())}")
    print("Metadata sample rates:", sorted(sample_frame.sample_rate.dropna().unique().tolist()))
    print("Duration values observed (s):", sample_frame.duration_seconds.min(), "to", sample_frame.duration_seconds.max())
    print("Observed sampled IDs include same segment_index across sets:", sample_frame.segment_index.duplicated(keep=False).any())
print("Audio checks:")
display(pd.DataFrame(audio_inspection))
print("Important limitation: passing checks only means the inspected examples passed.")

Rows inspected: 20
Duplicate (config, segment_index): 0
imlaey: null=0, empty=0
uthmani: null=0, empty=0
phonemes: null=0, empty=0
Metadata sample rates: [16000]
Duration values observed (s): 3.0199999809265137 to 6.28000020980835
Observed sampled IDs include same segment_index across sets: True
Audio checks:


,config,segment_index,http_status,bytes,declared_audio_type,decoded_rate,channels,duration_seconds,metadata_duration_seconds,finite,peak_abs
0,moshaf_0.1,001.0000,200,193964,audio/wav,16000,1,6.06,6.06,True,0.591248
1,moshaf_1.0,001.0000,200,156844,audio/wav,16000,1,4.90,4.90,True,0.474884


Important limitation: passing checks only means the inspected examples passed.


## 9. Findings and conclusions

- The dataset is organized into mushaf/recording-set configurations and has separate mushaf and reciter metadata configurations.
- The recitation examples are segmented audio records, with Quran text spans where applicable. Example rows also include isti`aatha outside the Quran text span.
- Imlaey and Uthmani text are separate; `phonemes` is a distinct phonetic representation, while `sifat` contains per-unit articulation/phonetic attributes.
- Several HF first-row requests fail with a server post-processing error. Counts are available from metadata, but a sample cannot be inspected there through this API in this run.
- This exploration does not verify every record or establish how the `match_ratio` was computed. The dataset card and original pipeline documentation should be read together with a broader Stage 2 audit before selecting targets.